# Phase 3 — fine-tune one class

Trains **one class per session**, resuming from NVIDIA's LSUN Dog net.

Uses exactly the configuration that completed Phase 2: **one GPU**, default
`batch_gpu`, five upstream patches. Nothing experimental.

## Run this as a saved version, not interactively

At 300 kimg this takes **~6.3 hours**. A browser tab will not survive that.

> **Save Version → Save & Run All (Commit)**

That runs headless for up to 12 h and keeps everything in the output. Set the
class in step 1 first. Accelerator **GPU T4 x2** (only one is used), Internet
**On**, with your `stylegan.zip` dataset attached.

## Order

| # | class | images | why |
|---|---|---|---|
| 1 | `mammalian` | 789 | most data, 37% of the image target alone |
| 2 | `arthropod` | 237 | already smoke-tested |
| 3 | `plant_fungus` | 197 | rounds the pilot out to 58% of target |

Three sessions, ~19 h total, inside one ~30 GPU-h week. Stop after these and
judge the results before spending quota on the other seven.

## 1. What to set

In [ ]:
CLASS   = "mammalian"    # then "arthropod", then "plant_fungus"
KIMG    = 300            # ~6.3 h at the measured 75 sec/kimg
FREEZED = 0              # FreezeD. First knob to try if results disappoint.
RESUME  = None           # None = start from LSUN Dog.
                         # To continue a previous session, give the path to its
                         # snapshot, e.g.
                         # "/kaggle/input/<dataset>/network-snapshot-000160.pkl"

print(f"{CLASS}: {KIMG} kimg  ~{KIMG * 75 / 3600:.1f} h at 75 sec/kimg")
print("12 h session cap ->", "fits" if KIMG * 75 / 3600 < 11 else "TOO LONG, lower KIMG")

## 2. Setup

The same five patches Phase 2 verified. `patch()` asserts its target exists, so
a silent no-op is impossible.

`conv2d_gradfix` stays disabled deliberately — its op was deleted from PyTorch,
and plain `F.conv2d` does double backward correctly. Its "Falling back" warning
is expected, and is filtered out of the training output below.

In [ ]:
import os, sys, json, time, pathlib, subprocess, shutil
import torch

REPO = "/kaggle/working/stylegan2-ada-pytorch"
shutil.rmtree(REPO, ignore_errors=True)
subprocess.run(["git", "clone", "-q",
                "https://github.com/NVlabs/stylegan2-ada-pytorch.git", REPO], check=True)
sys.path.insert(0, REPO)

def patch(rel, old, new):
    f = pathlib.Path(REPO) / rel
    s = f.read_text()
    assert old in s, f"patch target not found in {rel}"
    f.write_text(s.replace(old, new))

# custom_ops.py drops load()'s return value then re-imports by name.
patch("torch_utils/custom_ops.py",
      "torch.utils.cpp_extension.load(name=module_name",
      "module = torch.utils.cpp_extension.load(name=module_name")
patch("torch_utils/custom_ops.py",
      "        module = importlib.import_module(module_name)\n", "")

# PyTorch 2.x dropped Sampler.__init__(data_source).
patch("torch_utils/misc.py", "super().__init__(dataset)", "super().__init__()")

# grid_sample_gradfix supplies the second derivative of grid_sample, which torch
# lacks. R1 differentiates through the ADA augment pipeline twice, so without
# this, training dies at loss.py:131.
GSG = "torch_utils/ops/grid_sample_gradfix.py"
patch(GSG, "any(torch.__version__.startswith(x) for x in ['1.7.', '1.8.', '1.9'])",
      "True")
patch(GSG,
      "op = torch._C._jit_get_operation('aten::grid_sampler_2d_backward')\n"
      "        grad_input, grad_grid = op(grad_output, input, grid, 0, 0, False)",
      "op = torch.ops.aten.grid_sampler_2d_backward\n"
      "        grad_input, grad_grid = op(grad_output, input, grid, 0, 0, False, [True, True])")

print("torch", torch.__version__, "|", torch.cuda.get_device_name(0), "| 5 patches applied")

## 3. Dataset and source net

`dataset_tool.py` builds the zip, so the format cannot drift from what
`training/dataset.py` expects.

In [ ]:
DATA = next(p.parent for p in pathlib.Path("/kaggle/input").glob("**/summary.json"))
ZIP = f"/kaggle/working/{CLASS}.zip"
subprocess.run([sys.executable, f"{REPO}/dataset_tool.py",
                f"--source={DATA / CLASS}", f"--dest={ZIP}"], check=True)

URL = ("https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/"
       "transfer-learning-source-nets/lsundog-res256-paper256-kimg100000-noaug.pkl")
PKL = "/kaggle/working/lsundog-res256.pkl"
if not os.path.exists(PKL):
    subprocess.run(["wget", "-q", "-O", PKL, URL], check=True)

from training.dataset import ImageFolderDataset
ds = ImageFolderDataset(path=ZIP, use_labels=False, max_size=None, xflip=False)
print(f"{CLASS}: {len(ds)} images, {ds.image_shape}")
print("resuming from:", RESUME or "LSUN Dog (fresh fine-tune)")

## 4. Train

`--cfg=paper256` must match the source net or `--resume` fails on layer shapes.
`--snap=10` writes a snapshot and a sample grid every 40 kimg, so Phase 4 has
roughly eight checkpoints to choose between — the best is rarely the last.

`--metrics=none` on purpose. KID belongs in Phase 4, where it compares snapshots
in one pass; running it here spends time inside the 12 h cap on a number nobody
acts on until then.

In [ ]:
OUTDIR = f"/kaggle/working/{CLASS}_run"
cmd = [sys.executable, f"{REPO}/train.py",
       f"--outdir={OUTDIR}", f"--data={ZIP}", "--gpus=1",
       "--cfg=paper256", "--mirror=1", "--aug=ada", "--target=0.6",
       f"--resume={RESUME or PKL}", "--snap=10", "--metrics=none", f"--kimg={KIMG}"]
if FREEZED:
    cmd.append(f"--freezed={FREEZED}")

t0 = time.time()
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                     text=True, bufsize=1)
for line in p.stdout:
    if "conv2d_gradfix" not in line:
        print(line, end="")
p.wait()

if p.returncode != 0:
    raise SystemExit(f"training failed (exit {p.returncode})")
print(f"done in {(time.time()-t0)/3600:.2f} h")

## 5. Results

The sample grids are the honest signal. The deliverable is a game where people
guess real from fake, so **your eye is the metric that matters** — KID in Phase 4
only ranks snapshots that already look plausible.

`fakes_init.png` is the LSUN Dog starting point. Later grids should stop looking
like dogs and start looking like the class.

In [ ]:
import matplotlib.pyplot as plt
import PIL.Image

run = sorted(pathlib.Path(OUTDIR).glob("00000-*"))[-1]
grids = sorted(run.glob("fakes*.png"))
snaps = sorted(run.glob("network-snapshot-*.pkl"))

for g in [grids[0], grids[len(grids) // 2], grids[-1]]:
    im = PIL.Image.open(g)
    im.thumbnail((1100, 1100))
    plt.figure(figsize=(13, 13 * im.height / im.width))
    plt.imshow(im)
    plt.axis("off")
    plt.title(g.name)
    plt.show()

print(f"{len(snaps)} snapshots in {run.name}")
for s in snaps:
    print(f"  {s.name}  {s.stat().st_size / 1e6:.0f} MB")

ticks = [json.loads(l) for l in (run / "stats.jsonl").read_text().splitlines() if l.strip()]
get = lambda t, k: t.get(k, {}).get("mean", 0)
print()
for t in ticks[::max(1, len(ticks) // 10)]:
    print(f"  kimg {get(t,'Progress/kimg'):6.0f}  G {get(t,'Loss/G/loss'):7.3f}"
          f"  D {get(t,'Loss/D/loss'):7.3f}  ada_p {get(t,'Progress/augment'):.3f}")

## Done — then what

1. **Look at the last grid.** Do they read as creatures of this class?
2. If `ada_p` climbed past ~0.7, the discriminator was straining on too little
   data. Note it — that is the main argument for trying FreezeD next.
3. Repeat for the next class in the table at the top.

After all three, Phase 4 picks the best snapshot and truncation ψ per class and
screens for memorisation — a "fake" that is really a copy of a real Pokémon
would break the game outright.

Snapshots live in this notebook's **Output**. Keep at least the best one per
class; each is ~350 MB.